# Entrainement de 5 modeles\n\nNotebook d'experimentation pour comparer plusieurs modeles sur le dataset Kaggle `Health Insurance Cross Sell Prediction`.

## Configuration\n\n`SAMPLE_SIZE` controle le nombre de lignes utilisees pour l'experimentation.\n\n- `5_000` : test rapide\n- `50_000` : comparaison plus fiable\n- `0` : dataset complet

In [ ]:
import sys\nfrom pathlib import Path\n\nimport pandas as pd\nfrom IPython.display import Image, display\n\nROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()\nSRC = ROOT / "src"\nMODELS_DIR = ROOT / "models"\n\nif str(SRC) not in sys.path:\n    sys.path.insert(0, str(SRC))\n\nSAMPLE_SIZE = 5_000\nSELECTION_METRIC = "f1"\n\nROOT, SRC, MODELS_DIR, SAMPLE_SIZE, SELECTION_METRIC

## Validation du dataset

In [ ]:
from data import load_data\nfrom features import clean_data, expected_columns, validate_clean_data\n\nraw_df = load_data()\ndf = clean_data(raw_df)\nvalidate_clean_data(df)\n\nprint(f"Lignes brutes: {len(raw_df):,}")\nprint(f"Lignes nettoyees: {len(df):,}")\nprint(f"Colonnes utilisees: {expected_columns()}")\ndf.head()

In [ ]:
df["Response"].value_counts(normalize=True).rename("ratio").to_frame()

## Entrainement

In [ ]:
from train import train\n\nresults = train(sample_size=SAMPLE_SIZE, selection_metric=SELECTION_METRIC)\nresults

## Resultats sauvegardes

In [ ]:
metrics_path = MODELS_DIR / "metrics.csv"\npd.read_csv(metrics_path).sort_values("roc_auc", ascending=False)

In [ ]:
confusion_matrix_path = MODELS_DIR / "confusion_matrix.png"\ndisplay(Image(filename=str(confusion_matrix_path)))

## Lecture rapide\n\n- `roc_auc` mesure la capacite generale a separer les clients interesses des non interesses.\n- `recall` mesure la part de clients interesses retrouves par le modele.\n- `precision` mesure la qualite des clients selectionnes par le modele.\n- `f1` equilibre precision et recall.\n\nPour une campagne commerciale, le choix du modele ne depend pas seulement du `roc_auc` : un modele avec un meilleur `recall` peut etre preferable si l'objectif est de manquer le moins possible de clients interesses.